# Notebook 01 — Facial fungal adapter comparison (SD2.1 + cgDDI TI + custom LoRA)

**Goal:** Compare four configurations using **identical prompts and seeds**: (A) SD2.1 base, (B) SD2.1 + published `hcarrion/tinea_pedis` textual-inversion adapter, (C) SD2.1 + your own subtype LoRA, and (D) SD2.1 + published adapter + your LoRA.

**Critical evidence boundary:** `tinea_pedis` was trained on *athlete's foot*, not tinea faciei or facial pityriasis versicolor. The published adapter is therefore **only an exploratory transfer baseline**. Results MUST NOT be labeled clinically valid facial-fungal examples without expert inspection. The notebook intentionally prohibits its use on pityriasis versicolor. There is no verified dedicated public tinea-faciei or facial-versicolor diffusion checkpoint in the supplied research reports.

**Features:** Compel 2.3.1, `truncate_long_prompts=False`, up to 150 true content tokens across two SD2.1 CLIP chunks, positive/negative embedding padding, five output images per prompt per chosen configuration, separate image folders and metadata, ZIP export.

**Runtime:** Kaggle GPU T4 and Internet ON. The first two configurations can be run without a custom LoRA. Configure LoRA paths after training with Notebook 02. No existing LoRA weights are included.

**Run logs:** every generation attempt saves `notebook_console.log` (including tokenizer or CUDA errors) and `run_log.jsonl`. Successful outputs include per-image SHA-256 manifest, human-review draft CSV and summary JSON. Preserve all artifacts in Kaggle outputs; only privacy-reviewed aggregate statistics should be committed to public GitHub.


## 1. Install pinned packages
Run on a **fresh Kaggle T4 GPU notebook**. If you installed incompatible versions earlier in the session, restart before importing. Do not reinstall Kaggle's CUDA-enabled `torch`.

In [ ]:
import sys, subprocess
REQUIREMENTS = [
    'diffusers==0.35.1', 'transformers==4.56.2', 'accelerate==1.10.1',
    'peft==0.17.1', 'huggingface_hub==0.34.4', 'safetensors==0.6.2',
    'ftfy==6.3.1', 'compel==2.3.1',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *REQUIREMENTS])
print('Dependencies installed. Restart kernel and skip this cell if imports conflict with previously loaded packages.')

In [ ]:
import re, json, time, zipfile, gc, hashlib
from pathlib import Path
from PIL import Image
from IPython.display import display
import torch, diffusers, transformers
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from compel import Compel, ReturnedEmbeddingsType, DiffusersTextualInversionManager
from huggingface_hub import model_info
assert torch.cuda.is_available(), 'Choose GPU T4 in Kaggle Notebook Settings'
print('GPU:', torch.cuda.get_device_name(0), 'PyTorch:', torch.__version__)
print('Diffusers:', diffusers.__version__, 'Transformers:', transformers.__version__)

## 2. Configuration — edit here
The source identifier `hcarrion/tinea_pedis` is **not** a facial fungal LoRA. It is a published disease-concept textual inversion. Keep `TI_MODEL` as listed in cgDDI to test its transfer behavior. `LORA_PATHS` accepts the `.safetensors` files saved by the other notebook, or their containing directories.

`MODES_BY_SUBTYPE` determines comparisons to run. Initially, `tinea_faciei` tests **base + TI-only**, and pityriasis versicolor tests **base only**. After LoRA training, add `lora_only` and `ti_plus_lora` (tinea only).

In [ ]:
BASE_MODEL = 'sd2-community/stable-diffusion-2-1-base'  # community mirror; verify weight provenance
TI_MODEL = 'hcarrion/tinea_pedis'  # exact cgDDI listing: foot disease, not facial; EXPERIMENTAL TRANSFER ONLY
LORA_PATHS = {
    'tinea_faciei': '',                 # e.g. '/kaggle/input/tinea-lora/pytorch_lora_weights.safetensors'
    'pityriasis_versicolor': '',        # e.g. '/kaggle/input/pv-lora/pytorch_lora_weights.safetensors'
}
LORA_SCALES = {'tinea_faciei': 0.8, 'pityriasis_versicolor': 0.8}
MODES_BY_SUBTYPE = {
    'tinea_faciei': ['base', 'ti_only'],
    'pityriasis_versicolor': ['base'],
}
# AFTER LoRA training (uncomment desired modes, ensuring paths above are set):
# MODES_BY_SUBTYPE['tinea_faciei'] = ['base', 'ti_only', 'lora_only', 'ti_plus_lora']
# MODES_BY_SUBTYPE['pityriasis_versicolor'] = ['base', 'lora_only']

OUT_DIR = Path('/kaggle/working/fungal_adapter_comparison') / ('run_' + time.strftime('%Y%m%d_%H%M%S'))
IMAGES_PER_PROMPT = 5
HEIGHT = WIDTH = 512
STEPS = 30
GUIDANCE = 7.0
BASE_SEED = 62031
MAX_CONTENT_TOKENS = 150
MAX_CLIP_CHUNKS = 2
NEGATIVE_PROMPT = ('cartoon, illustration, CGI, heavy makeup, painting, waxy skin, '
                   'blurred face, distorted anatomy, duplicate faces, text, watermark')
ALLOWED_MODES = {'base', 'ti_only', 'lora_only', 'ti_plus_lora'}
assert all(0 <= v <= 2 for v in LORA_SCALES.values())
assert IMAGES_PER_PROMPT > 0 and HEIGHT % 8 == 0 and WIDTH % 8 == 0
assert MAX_CONTENT_TOKENS == 150 and MAX_CLIP_CHUNKS == 2
assert all(s in LORA_PATHS and set(modes) <= ALLOWED_MODES and modes for s,modes in MODES_BY_SUBTYPE.items())
assert all(not ('ti_only' in modes or 'ti_plus_lora' in modes) or s=='tinea_faciei'
           for s,modes in MODES_BY_SUBTYPE.items()), 'tinea_pedis TI cannot be assigned to pityriasis versicolor'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output folder:', OUT_DIR)

## 3. Load protected authored morphology prompts
Reads `prompts/` JSON files (300 prompts: 210 train, 45 val, 45 test; 60 independent families). Defaults to 5 train prompts; `PROMPT_LIMIT=None` selects all training prompts. Preserves original negative prompts and emits CSV+JSON review records. All generated images remain unapproved pending human QA.


In [ ]:
# Use project-authored 300 morphology prompts and protected prompt-family split.
from pathlib import Path
import sys,subprocess
DERMSYNREAL_REPO=Path('/kaggle/working/DermSynReal')
if not (DERMSYNREAL_REPO/'src/dermsynreal/kaggle_runs.py').is_file():
    subprocess.check_call(['git','clone','--depth','1','https://github.com/shanmukhdatta/DermSynReal.git',str(DERMSYNREAL_REPO)])
sys.path.insert(0,str(DERMSYNREAL_REPO/'src'))
from dermsynreal.kaggle_runs import prompt_collection,generation_manifest,append_run_log
PROMPT_PARTITION='train'
PROMPT_VARIANT='morphology'  # use 'generic' for E1, morphology for E2
PROMPT_OFFSET=0
PROMPT_LIMIT=5   # Kaggle pilot; None uses all 210 train prompts
PROMPTS=prompt_collection(DERMSYNREAL_REPO,'Fungal_Infection',partition=PROMPT_PARTITION,limit=PROMPT_LIMIT,offset=PROMPT_OFFSET,variant=PROMPT_VARIANT)
# Keep E1 generic and E2 morphology images in separate run directories; never overwrite.
OUT_DIR=Path(OUT_DIR)/('prompt_'+PROMPT_VARIANT)
if OUT_DIR.exists():
    raise FileExistsError(f'Generation output already exists: {OUT_DIR}; start a fresh run directory')
print('Authored prompts:',len(PROMPTS),'families:',len({p['prompt_family_id'] for p in PROMPTS}))
# Candidate unconfirmed candida appearance: base-only, requires review before training.
MODES_BY_SUBTYPE.setdefault('unconfirmed_candida',['base'])


In [ ]:
# Capture generation console, warnings and failed diffusion execution to disk.
from dermsynreal.kaggle_runs import start_notebook_logging
GENERATION_CONSOLE=start_notebook_logging(OUT_DIR,kind='diffusion_generation',
    config={'prompt_partition':PROMPT_PARTITION,'prompt_variant':PROMPT_VARIANT,
            'prompt_count':len(PROMPTS),'class_name':'Fungal_Infection',
            'images_per_prompt':IMAGES_PER_PROMPT},repo_root=DERMSYNREAL_REPO)
print('Generation console file:',GENERATION_CONSOLE)


## 4. Load SD2.1, published TI, and any available custom LoRA weights
The exact **published TI trigger is discovered** by comparing tokenizer additions before and after loading; the notebook does not guess `<tinea-pedis-class>`. If the model resource cannot be downloaded or lacks compatible textual-inversion weights, this cell stops with a clear explanation instead of pretending to run the adapter.

The public TI and custom LoRAs are separate components. Your own LoRA is trained on SD2.1 in Notebook 02 and is combined at **inference**, not initialized from the published TI.

In [ ]:
base_revision = model_info(BASE_MODEL).sha
needs_ti = any(m in {'ti_only','ti_plus_lora'} for modes in MODES_BY_SUBTYPE.values() for m in modes)
needs_lora = {s for s,modes in MODES_BY_SUBTYPE.items() if any(m in {'lora_only','ti_plus_lora'} for m in modes)}
for subtype in sorted(needs_lora):
    if not str(LORA_PATHS[subtype]).strip():
        raise RuntimeError(f'{subtype}: LoRA experiment requested but LORA_PATHS has no checkpoint path')

pipe = StableDiffusionPipeline.from_pretrained(
    BASE_MODEL, revision=base_revision, torch_dtype=torch.float16, use_safetensors=True,
)
public_revision = None
TI_TRIGGER = None
if needs_ti:
    public_revision = model_info(TI_MODEL).sha
    before_tokens = set(pipe.tokenizer.get_added_vocab())
    try:
        pipe.load_textual_inversion(TI_MODEL, revision=public_revision)
    except Exception as error:
        raise RuntimeError(
            'Could not load hcarrion/tinea_pedis as a compatible textual inversion. '
            'Check model files, availability and SD2.1 architecture on Hugging Face. '
            'Do not replace it with a guessed LoRA.') from error
    after_tokens = set(pipe.tokenizer.get_added_vocab())
    new_tokens = sorted(after_tokens - before_tokens)
    # Multivector TI can register <concept>, <concept>_1, ... ; select unsuffixed root.
    root_tokens = [t for t in new_tokens if not re.search(r'_\d+$', t)]
    if len(root_tokens) != 1:
        raise RuntimeError(f'Cannot unambiguously identify published TI trigger. New tokens: {new_tokens!r}')
    TI_TRIGGER = root_tokens[0]
    print('Published TI loaded:', TI_MODEL, '| exact trigger discovered:', TI_TRIGGER,
          '| revision:', public_revision)

loaded_loras = {}
for subtype in sorted(needs_lora):
    path = str(LORA_PATHS[subtype]).strip()
    local = Path(path)
    if local.exists():
        provenance_dir = local.parent if local.is_file() else local
        provenance_file = provenance_dir/'training_provenance.json'
        if provenance_file.exists():
            provenance = json.loads(provenance_file.read_text(encoding='utf-8'))
            if provenance.get('training_completed') is False:
                raise RuntimeError(f'{subtype}: training provenance says LoRA incomplete')
            for name, expected in [('base_model',BASE_MODEL),('base_revision',base_revision),('subtype',subtype)]:
                if provenance.get(name) != expected:
                    raise RuntimeError(f'{subtype}: provenance mismatch on {name}')
            weights_path = local if local.is_file() else local/'pytorch_lora_weights.safetensors'
            if weights_path.is_file() and provenance.get('lora_sha256'):
                if hashlib.sha256(weights_path.read_bytes()).hexdigest() != provenance['lora_sha256']:
                    raise RuntimeError(f'{subtype}: LoRA checksum mismatch')
            print('Validated training provenance:', subtype)
        else:
            print('WARNING:', subtype, 'has no training_provenance.json; verify base model manually')
    if local.is_file():
        if local.suffix != '.safetensors':
            raise ValueError(f'Expected .safetensors: {local}')
        pipe.load_lora_weights(str(local.parent), weight_name=local.name, adapter_name=subtype)
    else:
        pipe.load_lora_weights(path, adapter_name=subtype)
    loaded_loras[subtype] = path

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.enable_vae_slicing()
pipe.to('cuda')
pipe.text_encoder.eval()
print('Stable Diffusion base loaded:', BASE_MODEL, base_revision)
print('Custom LoRAs loaded:', list(loaded_loras))

## 5. Strict Compel 150-content-token validation and TI support
SD2.1 CLIP uses **77 positions including BOS/EOS**, leaving 75 content tokens in each chunk. Compel can encode two chunks. **Both positive and negative** prompts must fit the 150-content-token design limit. Multi-vector textual-inversion expansion is counted **after** prompt conversion.

`DiffusersTextualInversionManager` is passed to Compel so a public TI token is expanded correctly. Compel runs with `truncate_long_prompts=False`. To avoid Compel parsing special weighted syntax unpredictably, ordinary natural-language prompts are recommended.

In [ ]:
assert pipe.tokenizer.model_max_length == 77, 'Expected 77-position SD2.1 tokenizer'
CONTENT_PER_CHUNK = pipe.tokenizer.model_max_length - 2
assert CONTENT_PER_CHUNK * MAX_CLIP_CHUNKS == MAX_CONTENT_TOKENS

encoder = Compel(
    tokenizer=pipe.tokenizer,
    text_encoder=pipe.text_encoder,
    textual_inversion_manager=DiffusersTextualInversionManager(pipe) if needs_ti else None,
    returned_embeddings_type=ReturnedEmbeddingsType.LAST_HIDDEN_STATES_NORMALIZED,
    truncate_long_prompts=False,
    requires_pooled=False,
    device='cuda',
)

def actual_token_count(prompt):
    expanded = pipe.maybe_convert_prompt(prompt, pipe.tokenizer) if needs_ti else prompt
    return len(pipe.tokenizer(expanded, truncation=False, add_special_tokens=False)['input_ids'])

def require_token_budget(prompt, label):
    if not isinstance(prompt,str) or not prompt.strip():
        raise ValueError(f'{label}: empty text')
    count = actual_token_count(prompt)
    if count > MAX_CONTENT_TOKENS:
        raise ValueError(f'{label}: {count} actual content tokens exceeds limit {MAX_CONTENT_TOKENS}')
    return count

def positive_prompt(item, experiment_mode):
    base = item['text'].strip()
    if experiment_mode in ('ti_only','ti_plus_lora'):
        if not TI_TRIGGER:
            raise RuntimeError('TI experiment selected but public TI was not loaded')
        if TI_TRIGGER in base:
            raise ValueError('Do not manually include the public TI trigger; it is appended automatically')
        return base + ', ' + TI_TRIGGER
    if TI_TRIGGER and TI_TRIGGER in base:
        raise ValueError('Base/LoRA-only prompt cannot contain pretrained TI trigger')
    return base

for item in PROMPTS:
    for mode in MODES_BY_SUBTYPE[item['subtype']]:
        prompt = positive_prompt(item, mode)
        count = require_token_budget(prompt, f"{item['id']}/{mode}")
        print(item['id'], mode, 'token count', count, 'chunk estimate', max(1,(count+74)//75))
for item in PROMPTS:
    require_token_budget(item.get('negative_prompt',NEGATIVE_PROMPT),item['id']+'/negative')

## 6. Generate five images per prompt **per configuration**
Switching adapters is explicit. For base/TI-only experiments, custom LoRA weights are set to zero (rather than accidentally left active). TI conditioning is activated only when the discovered trigger is present in the prompt.

Images are previewed and stored under `prompt_id / experiment_mode /`. A common seed is used for each prompt across all selected configurations, enabling side-by-side comparison. This is not proof of clinical fidelity.

In [ ]:
def safe_id(value):
    return re.sub(r'[^a-zA-Z0-9_-]+','_',str(value)).strip('_')[:70] or 'prompt'

def activate_lora(subtype, experiment_mode):
    if experiment_mode in ('lora_only','ti_plus_lora'):
        if subtype not in loaded_loras:
            raise RuntimeError(f'Custom LoRA missing for {subtype}')
        pipe.set_adapters([subtype], adapter_weights=[LORA_SCALES[subtype]])
        return LORA_SCALES[subtype]
    # If at least one LoRA is loaded, explicitly disable all during base/TI-only modes.
    if loaded_loras:
        for name in loaded_loras:
            pipe.set_adapters([name], adapter_weights=[0.0])
    return 0.0

all_metadata = []
started = time.time()
for index, item in enumerate(PROMPTS):
    subtype = item['subtype']
    for experiment_mode in MODES_BY_SUBTYPE[subtype]:
        if experiment_mode in ('ti_only','ti_plus_lora') and subtype != 'tinea_faciei':
            raise RuntimeError('Published tinea_pedis TI must not be treated as a versicolor adapter')
        scale = activate_lora(subtype, experiment_mode)
        prompt = positive_prompt(item, experiment_mode)
        tokens = require_token_budget(prompt, f"{item['id']}/{experiment_mode}")
        negative_tokens = require_token_budget(item.get('negative_prompt',NEGATIVE_PROMPT), 'negative')
        with torch.no_grad():
            positive_embeds = encoder(prompt)
            negative_embeds = encoder(item.get('negative_prompt',NEGATIVE_PROMPT))
            positive_embeds, negative_embeds = encoder.pad_conditioning_tensors_to_same_length(
                [positive_embeds, negative_embeds]
            )
        if positive_embeds.shape != negative_embeds.shape or positive_embeds.ndim != 3:
            raise RuntimeError(f'Invalid Compel shapes: {positive_embeds.shape}, {negative_embeds.shape}')
        if positive_embeds.shape[1] > MAX_CLIP_CHUNKS * pipe.tokenizer.model_max_length:
            raise RuntimeError(f'Compel produced more than two CLIP chunks: {positive_embeds.shape}')
        positive_embeds = positive_embeds.to(device='cuda',dtype=pipe.unet.dtype)
        negative_embeds = negative_embeds.to(device='cuda',dtype=pipe.unet.dtype)
        dst = OUT_DIR / safe_id(item['id']) / experiment_mode
        dst.mkdir(parents=True,exist_ok=True)
        preview_images = []
        for number in range(IMAGES_PER_PROMPT):
            seed = BASE_SEED + index*1000 + number
            rng = torch.Generator(device='cuda').manual_seed(seed)
            with torch.inference_mode():
                generated = pipe(
                    prompt_embeds=positive_embeds, negative_prompt_embeds=negative_embeds,
                    width=WIDTH,height=HEIGHT,num_inference_steps=STEPS,
                    guidance_scale=GUIDANCE,generator=rng,
                ).images[0]
            output = dst / f'{safe_id(item["id"])}_{experiment_mode}_seed{seed}.png'
            generated.save(output)
            preview_images.append(generated)
            all_metadata.append({
                'file':output.relative_to(OUT_DIR).as_posix(),
                'prompt_id':item['id'], 'subtype_requested':subtype,
                'experiment_mode':experiment_mode, 'prompt':prompt,
                'negative_prompt':item.get('negative_prompt',NEGATIVE_PROMPT), 'prompt_content_tokens':tokens,
                'negative_content_tokens':negative_tokens,
                'conditioning_positions':positive_embeds.shape[1],
                'seed':seed,'base_model':BASE_MODEL,'base_revision':base_revision,
                'published_ti':TI_MODEL if experiment_mode in ('ti_only','ti_plus_lora') else None,
                'published_ti_revision':public_revision if experiment_mode in ('ti_only','ti_plus_lora') else None,
                'published_ti_trigger':TI_TRIGGER if experiment_mode in ('ti_only','ti_plus_lora') else None,
                'custom_lora':loaded_loras.get(subtype) if scale else None,
                'custom_lora_scale':scale,'width':WIDTH,'height':HEIGHT,
                'steps':STEPS,'guidance':GUIDANCE,
                'clinical_review':'pending','synthetic_use':'exploratory_only',
                'ti_domain_mismatch_warning':experiment_mode in ('ti_only','ti_plus_lora'),
            })
        montage=Image.new('RGB',(WIDTH*len(preview_images),HEIGHT))
        for k,im in enumerate(preview_images):montage.paste(im,(k*WIDTH,0))
        show_width=min(1200,montage.width)
        print(item['id'],'|',experiment_mode,'|',len(preview_images),'images')
        display(montage.resize((show_width,round(HEIGHT*show_width/montage.width))))
        del positive_embeds,negative_embeds
        gc.collect()

with (OUT_DIR/'metadata.jsonl').open('w',encoding='utf-8') as f:
    for entry in all_metadata:
        f.write(json.dumps(entry,ensure_ascii=False)+'\n')
print('Generated:',len(all_metadata),'| Output:',OUT_DIR,'| Seconds:',round(time.time()-started,1))

In [ ]:
# Export reproducible per-image CSV, SHA hashes, review draft and run provenance.
summary=generation_manifest(all_metadata,OUT_DIR/'reports',root=OUT_DIR,class_name='Fungal_Infection',repo_root=DERMSYNREAL_REPO,prompt_partition=PROMPT_PARTITION,model_id=BASE_MODEL,prompt_type=PROMPT_VARIANT)
append_run_log(OUT_DIR,kind='generation',config={'class':'Fungal_Infection','prompt_partition':PROMPT_PARTITION,'prompt_type':PROMPT_VARIANT,'prompt_count':len(PROMPTS),'images_per_prompt':IMAGES_PER_PROMPT,'generated':summary['generated_images']},artifacts=[OUT_DIR/'reports/generation_manifest.csv',OUT_DIR/'metadata.jsonl'],repo_root=DERMSYNREAL_REPO)
print('CSV:',OUT_DIR/'reports/generation_manifest.csv','Review draft:',OUT_DIR/'reports/synthetic_pool_review_draft.csv')

## 7. Export one ZIP, with filenames and metadata
Download via Kaggle's Output panel. Preserve the experiment mode and seed in any downstream comparisons. These samples are **not validated medical labels**.

In [ ]:
ZIP_PATH = Path('/kaggle/working') / (OUT_DIR.name+'_fungal_adapter_comparison.zip')
with zipfile.ZipFile(ZIP_PATH,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as archive:
    for p in sorted(OUT_DIR.rglob('*')):
        if p.is_file():archive.write(p,arcname=p.relative_to(OUT_DIR).as_posix())
expected=sum(len(MODES_BY_SUBTYPE[p['subtype']]) for p in PROMPTS)*IMAGES_PER_PROMPT
assert len(all_metadata)==expected, f'Expected {expected} images, got {len(all_metadata)}'
assert len(list(OUT_DIR.rglob('*.png')))==expected
print('Download:',ZIP_PATH, '| MiB:',round(ZIP_PATH.stat().st_size/2**20,2))

## Sources and limits
- cgDDI official code and released adapter list: https://github.com/hectorcarrion/ControllableGenDDI
- Report-listed fungal concept: https://huggingface.co/hcarrion/tinea_pedis (foot infection only)
- Diffusers textual inversion loader documentation: https://huggingface.co/docs/diffusers/main/api/loaders
- Compel 2.3.1: https://pypi.org/project/compel/2.3.1/

No GPU run was performed in the offline audit. Do not use `tinea_pedis`-conditioned facial images as clinically grounded tinea-faciei training data without independent expert morphology review and memorization checks.